[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajitpanday80/ai-learn/blob/main/notebooks/llm-train-scratch.ipynb)

# 🧠 How to Train an LLM from Scratch

**Section:** LLMs

Models like GPT, Claude and Llama all come out of the same basic recipe. In this lesson you'll run that whole recipe on a small scale and build a **tiny GPT** that learns to write Shakespeare-style text one character at a time.

### The pretraining pipeline
| Step | What happens | In this notebook |
|---|---|---|
| 1. **Data** | Collect and clean a large text corpus | Tiny Shakespeare (~1 MB) |
| 2. **Tokenizer** | Turn text into integer IDs | Character-level (65 tokens) |
| 3. **Architecture** | Decoder-only Transformer | 4 layers, ~1.8M parameters |
| 4. **Objective** | Predict the next token (cross-entropy) | Same as the real models |
| 5. **Optimization** | AdamW, LR warmup + cosine decay, gradient clipping | Same as the real models |
| 6. **Sampling** | Generate text one token at a time | Temperature and top-k |

The training objective is simple: given tokens $x_1, \dots, x_t$, maximize the probability of $x_{t+1}$. Grammar, facts and style all get learned as a side effect of getting better at that one task.

> ⚡ **Use a GPU:** In Colab go to **Runtime > Change runtime type > GPU** (T4 is fine). On a GPU, training takes about 2 minutes. On a CPU it still runs, but lower `max_iters` first.

In [ ]:
!pip install -q torch matplotlib

import math, time, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.style.use('dark_background')
torch.manual_seed(1337)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)
if device == 'cpu':
    print('⚠️  No GPU found. Runtime > Change runtime type > GPU gives roughly 10x faster training.')

# ---- Step 1: Data ----
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
text = urllib.request.urlopen(url).read().decode('utf-8')
print(f'Corpus size: {len(text):,} characters\n')
print(text[:300])

## 🔤 Tokenization and 🏗️ Architecture

**Tokenizer.** Real LLMs use subword tokenizers (BPE) with vocabularies of 32k–200k tokens. We use one token per character to keep things simple. The model is the same either way; only the vocabulary size changes.

**Decoder-only Transformer (GPT).** Each block contains:
1. **Causal self-attention.** Each position can look at earlier positions only, never later ones. This is what makes next-token prediction a fair task.
2. **MLP.** A per-position feed-forward network (4x expansion with GELU).
3. **Residual connections + pre-LayerNorm.** These keep deep networks trainable.

Token embeddings plus position embeddings go in, and logits over the vocabulary come out.

🧪 **Experiment:** change the `config` dict below (more layers, a wider `n_embd`, a longer `block_size`) and watch how the parameter count and final loss change.

In [ ]:
# ---- Step 2: Tokenizer ----
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

print('Vocab size:', vocab_size)
print('encode("Hello") ->', encode('Hello'))
print('decode(...)    ->', decode(encode('Hello')))

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

# 🧪 EXPERIMENT: model hyperparameters
config = dict(
    block_size=128,   # context length (tokens the model can see)
    n_embd=192,       # embedding / hidden width
    n_head=6,         # attention heads (must divide n_embd)
    n_layer=4,        # transformer blocks
    dropout=0.1,
)
batch_size = 64

def get_batch(split):
    d = train_data if split == 'train' else val_data
    bs = config['block_size']
    ix = torch.randint(len(d) - bs - 1, (batch_size,))
    x = torch.stack([d[i:i + bs] for i in ix])
    y = torch.stack([d[i + 1:i + bs + 1] for i in ix])  # targets = inputs shifted by one
    return x.to(device), y.to(device)

# ---- Step 3: Architecture ----
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        # is_causal=True applies the mask so tokens cannot see the future
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd, n_head, n_layer, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying, as in GPT-2
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        x = self.blocks(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

model = TinyGPT(vocab_size, **config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'\nModel parameters: {n_params/1e6:.2f}M')

# Sanity check: an untrained model should be close to uniform, so loss is about ln(vocab_size)
xb, yb = get_batch('train')
_, loss0 = model(xb, yb)
print(f'Initial loss: {loss0.item():.3f}  (expected about ln({vocab_size}) = {math.log(vocab_size):.3f})')

## 🏋️ Pretraining

The training loop below is structurally the same one used for frontier models, just much smaller:
- **AdamW** with weight decay and betas `(0.9, 0.95)`, a common choice for LLM training
- **Linear warmup, then cosine decay** of the learning rate, so early noisy gradients don't blow up training
- **Gradient clipping** at 1.0 to guard against loss spikes
- **Periodic validation**, so overfitting shows up as a gap between train and val loss

🧪 **Experiments to try:**
- Set `learning_rate = 1e-2` and watch training become unstable. Set it to `1e-4` and watch it crawl.
- Set `warmup_iters = 0`.
- Increase `max_iters` to 5000 and see whether validation loss keeps improving or starts to overfit.

In [ ]:
# 🧪 EXPERIMENT: training hyperparameters
max_iters     = 2000 if device == 'cuda' else 300
eval_interval = 200 if device == 'cuda' else 50
eval_iters    = 50
learning_rate = 1e-3
warmup_iters  = 100
min_lr        = learning_rate / 10

def get_lr(it):
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    progress = (it - warmup_iters) / max(1, max_iters - warmup_iters)
    return min_lr + 0.5 * (learning_rate - min_lr) * (1 + math.cos(math.pi * progress))

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,
                              betas=(0.9, 0.95), weight_decay=0.1)

step_losses, lrs, eval_steps, train_evals, val_evals = [], [], [], [], []
t0 = time.time()
model.train()
for it in range(max_iters + 1):
    lr = get_lr(it)
    for g in optimizer.param_groups:
        g['lr'] = lr

    if it % eval_interval == 0:
        l = estimate_loss()
        eval_steps.append(it); train_evals.append(l['train']); val_evals.append(l['val'])
        print(f'step {it:5d} | train {l["train"]:.3f} | val {l["val"]:.3f} | lr {lr:.2e} | {time.time()-t0:.0f}s')

    x, y = get_batch('train')
    _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    step_losses.append(loss.item()); lrs.append(lr)

# ---- Visualize training ----
def smooth(xs, k=25):
    return [sum(xs[max(0, i-k):i+1]) / len(xs[max(0, i-k):i+1]) for i in range(len(xs))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(step_losses, color='#4C9AFF', alpha=0.25, linewidth=1)
ax1.plot(smooth(step_losses), color='#4C9AFF', linewidth=2, label='train (per step, smoothed)')
ax1.plot(eval_steps, val_evals, 'o-', color='#FF8A4C', linewidth=2, label='validation')
ax1.axhline(math.log(vocab_size), color='gray', linestyle='--', linewidth=1, label='random guessing')
ax1.set_title('Cross-entropy loss'); ax1.set_xlabel('step'); ax1.set_ylabel('loss')
ax1.legend(frameon=False); ax1.grid(alpha=0.15)

ax2.plot(lrs, color='#5CD6A0', linewidth=2)
ax2.set_title('Learning rate: warmup + cosine decay'); ax2.set_xlabel('step'); ax2.set_ylabel('lr')
ax2.grid(alpha=0.15)
plt.tight_layout(); plt.show()

print(f'Final val loss {val_evals[-1]:.3f}, perplexity {math.exp(val_evals[-1]):.2f} '
      f'(a random model would score {vocab_size})')

In [ ]:
# ---- Step 6: Sampling ----
# 🧪 EXPERIMENT: change the prompt, temperature, and top_k
prompt = 'ROMEO:'
temperatures = [0.5, 1.0, 1.5]
top_k = 40
n_chars = 300

def sample(m, prompt, temperature, top_k, n):
    m.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    out = m.generate(idx, n, temperature=temperature, top_k=top_k)
    return decode(out[0].tolist())

# Before vs. after: an untrained model with the same architecture
untrained = TinyGPT(vocab_size, **config).to(device)
print('=' * 60, '\n❌ UNTRAINED MODEL\n' + '=' * 60)
print(sample(untrained, prompt, 1.0, None, 150))

for t in temperatures:
    print('\n' + '=' * 60, f'\n✅ TRAINED MODEL | temperature={t} | top_k={top_k}\n' + '=' * 60)
    print(sample(model, prompt, t, top_k, n_chars))

# Visualize the next-token distribution after the prompt
model.eval()
with torch.no_grad():
    idx = torch.tensor([encode(prompt)], device=device)
    logits, _ = model(idx)
    probs = F.softmax(logits[0, -1], dim=-1).cpu()
top_p, top_i = probs.topk(15)
labels = [repr(itos[i.item()]) for i in top_i]
plt.figure(figsize=(11, 3.5))
plt.bar(labels, top_p.numpy(), color='#4C9AFF')
plt.title(f'Top-15 next-token probabilities after {prompt!r}')
plt.ylabel('probability'); plt.grid(axis='y', alpha=0.15)
plt.tight_layout(); plt.show()

## 🚀 From TinyGPT to a real LLM

You just ran the core of LLM pretraining. Production models use the same recipe; mostly the scale changes:

| | This notebook | Frontier-scale LLM |
|---|---|---|
| Parameters | ~1.8M | 10B – 1T+ |
| Training tokens | ~1M characters | 10T+ tokens |
| Tokenizer | 65 characters | BPE, 100k+ vocab |
| Context | 128 | 8k – 1M+ |
| Hardware | 1 GPU, minutes | Thousands of GPUs, months |
| Architecture tweaks | LayerNorm, learned positions | RMSNorm, RoPE, SwiGLU, GQA, sometimes MoE |

**Key ideas to take away:**
- **Scaling laws.** Loss falls predictably as parameters, data and compute grow. The *Chinchilla* result suggests about **20 training tokens per parameter** for a compute-optimal model.
- **Data quality matters as much as quantity.** Real pipelines spend a lot of effort on deduplication, filtering and choosing the data mix.
- **Pretraining produces a base model.** A base model continues text; it doesn't follow instructions. Turning it into an assistant takes **supervised fine-tuning (SFT)** followed by **preference optimization (RLHF / DPO)**.

### 🧪 Challenges
1. **Scale up.** Set `n_layer=6, n_embd=384` and `max_iters=5000`. How far does val loss drop? Does overfitting appear?
2. **Scale down.** Set `n_layer=1`. What does the text look like now?
3. **Context length.** Try `block_size=32` and then `256`. How does the output's coherence change?
4. **New corpus.** Replace `url` with any plain-text file (for example, a Project Gutenberg book) and train on it.
5. **Better tokenizer.** Run `!pip install tiktoken` and use GPT-2's BPE (`tiktoken.get_encoding('gpt2')`) in place of characters. What happens to the vocab size and to training speed?